In [0]:
USE CATALOG tibia_analytics;
USE SCHEMA silver;
SET TIME ZONE 'UTC';

In [0]:
MERGE INTO silver.worlds_summary AS target
USING (
  SELECT
    raw_worlds_id AS snapshot_id,
    worlds.players_online AS players_online,
    worlds.record_players AS record_players,
    to_timestamp(worlds.record_date) AS record_at,
    information.api.version AS api_version,
    information.api.release AS api_release,
    information.api.commit AS api_commit,
    to_timestamp(information.timestamp) AS api_processed_at,
    information.tibia_urls AS tibia_urls,
    information.status.error AS error,
    information.status.http_code AS http_code,
    information.status.message AS message,
    source_file_date,
    source_file_path,
    source_file_modified_at,
    ingested_at,
    current_timestamp() AS processed_at
  FROM tibia_analytics.bronze.raw_worlds
) AS source
ON target.snapshot_id = source.snapshot_id
WHEN NOT MATCHED THEN
  INSERT *;

In [0]:
MERGE INTO silver.worlds AS target
USING (
  SELECT
    raw_worlds_id AS snapshot_id,
    sha1(lower(trim(world.name))) AS world_id,
    world.name AS world,
    world.status AS status,
    world.players_online AS players_online,
    world.location AS location,
    world.pvp_type AS pvp_type,
    world.premium_only AS premium_only,
    world.transfer_type AS transfer_type,
    world.battleye_protected AS battleye_protected,
    to_date(
      CASE
        WHEN lower(trim(world.battleye_date)) = 'release' THEN null
        else world.battleye_date
      END
    ) AS battleye_date,
    world.game_world_type AS world_type,
    NULL AS tournament_type,
    source_file_date,
    ingested_at,
    current_timestamp() AS processed_at
  FROM tibia_analytics.bronze.raw_worlds
  LATERAL VIEW explode(worlds.regular_worlds) worlds_table AS world
  UNION ALL
  SELECT
    raw_worlds_id AS snapshot_id,
    sha1(lower(trim(tournament.name))) AS world_id,
    tournament.name AS world,
    tournament.status AS status,
    tournament.players_online AS players_online,
    tournament.location AS location,
    tournament.pvp_type AS pvp_type,
    tournament.premium_only AS premium_only,
    tournament.transfer_type AS transfer_type,
    tournament.battleye_protected AS battleye_protected,
    to_date(
      CASE
        WHEN lower(trim(tournament.battleye_date)) = 'release' THEN null
        else tournament.battleye_date
      END
    ) AS battleye_date,
    tournament.game_world_type AS world_type,
    tournament.tournament_world_type AS tournament_type,
    source_file_date,
    ingested_at,
    current_timestamp() AS processed_at
  FROM tibia_analytics.bronze.raw_worlds
  LATERAL VIEW explode(worlds.tournament_worlds) worlds_table AS tournament
) AS source
ON target.snapshot_id = source.snapshot_id
WHEN NOT MATCHED THEN
  INSERT *;